<a href="https://colab.research.google.com/github/abxda/portable-satelital/blob/main/colab/Taller_ML_Urbano_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 🛰️ Taller: Machine Learning para detectar zonas urbanas — **en Google Colab**

Esta es la versión para **Google Colab** del taller del curso. Hace exactamente lo mismo que la versión del navegador (WASM), pero aquí corre sobre **Python real en la nube de Google**: la segmentación de todo Aguascalientes tarda ~1 minuto en lugar de ~3.

El motor de segmentación es **`pyshepseg`** (la librería original de la que `shepherd-wasm` es un port a WebAssembly, validado bit a bit). El pipeline de Machine Learning es idéntico al del curso.

> ▶️ **Cómo correrlo:** menú **Entorno de ejecución → Ejecutar todas**, o celda por celda con **Shift+Enter**.


## Paso 0 — Instalar las herramientas

Colab ya trae NumPy, SciPy, scikit-learn y Matplotlib. Solo agregamos las geoespaciales (`rasterio`, `geopandas`) y el segmentador `pyshepseg`. Tarda ~1 minuto la primera vez.


In [ ]:
# pyshepseg vive en GitHub (no en PyPI); rasterio y geopandas leen/escriben geodatos
!pip install -q "git+https://github.com/ubarsc/pyshepseg.git" rasterio geopandas
print("✓ herramientas listas")

## Paso 1a — Traer la imagen del estado completo

Una imagen **calidad Landsat** (30 m, 6 bandas) de todo Aguascalientes, derivada de una geomediana Sentinel-2. La descargamos una sola vez desde Hugging Face.


In [ ]:
import os, time, urllib.request
import numpy as np
import rasterio
from rasterio import features
import matplotlib.pyplot as plt
from scipy import ndimage
from pyshepseg import shepseg

HF = "https://huggingface.co/datasets/abxda/portable-satelital/resolve/main/taller/"

def carga_dato(nombre):
    """Descarga un dato del taller (una sola vez) desde Hugging Face."""
    if not os.path.exists(nombre):
        print(f"  descargando {nombre} ...")
        urllib.request.urlretrieve(HF + nombre, nombre)
    return nombre

t0 = time.time()
ruta = carga_dato("ags_landsat_30m.tif")
with rasterio.open(ruta) as src:
    img = src.read()                  # (bandas, filas, columnas) — ENTERO (lo que pide pyshepseg)
    transform, crs, nodata = src.transform, src.crs, src.nodata
    nombres_bandas = src.descriptions
nodata = int(nodata)                  # -9999: valor de "sin dato" (fuera del estado)

n_bandas, alto, ancho = img.shape
print(f"✓ imagen lista en {time.time()-t0:.1f} s: {n_bandas} bandas × {alto} × {ancho}  (dtype {img.dtype})")
print(f"  cada píxel cubre 30 m → la escena mide ≈ {ancho*30/1000:.0f} × {alto*30/1000:.0f} km: todo Aguascalientes")
print(f"  bandas: {', '.join(nombres_bandas)}")
print("💡 Para la máquina, la 'imagen' son 73 millones de números enteros. Nada más.")

## Paso 1b — Verla como la verían tus ojos (color natural)


In [ ]:
# Bandas rojo(b3), verde(b2), azul(b1) → color natural; enmascaramos el "sin dato"
rgb = np.stack([img[2], img[1], img[0]], axis=-1).astype(np.float32)
mask = (img[0] == nodata)                            # píxeles fuera del estado
rgb[mask] = np.nan
p2, p98 = np.nanpercentile(rgb[::5, ::5], (2, 98))   # contraste: percentiles 2–98
rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)
rgb = np.nan_to_num(rgb, nan=1.0)                    # sin dato → blanco

plt.figure(figsize=(7.5, 7))
plt.imshow(rgb[::3, ::3])                             # 1 de cada 3 píxeles para dibujar rápido
plt.title("Aguascalientes completo — 'calidad Landsat' (30 m)")
plt.axis("off"); plt.show()

## Paso 1 (segmentación) — Siembra: K-means agrupa por color


In [ ]:
t0 = time.time()
# K-means agrupa los píxeles por su firma espectral, SIN mirar dónde están:
#   numClusters=60  → cuántas "familias espectrales" buscar
#   subsamplePcnt=1 → ajusta con 1% de los píxeles (rápido y suficiente)
#   fixedKMeansInit → inicio fijo: resultado reproducible
km = shepseg.fitSpectralClusters(img, 60, 1, nodata, True)
clusters = shepseg.applySpectralClusters(km, img, nodata)   # a CADA píxel su familia
print(f"K-means listo en {time.time()-t0:.1f} s")

plt.figure(figsize=(12, 5.5))
plt.subplot(1, 2, 1); plt.imshow(rgb[::3, ::3]); plt.title("El estado"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(clusters[::3, ::3], cmap="tab20", interpolation="nearest")
plt.title("Familias espectrales (colores = familias)"); plt.axis("off")
plt.tight_layout(); plt.show()
print("💡 Las familias capturan tipos de cobertura, pero quedan 'salpicadas' (sal y pimienta).")

## Pasos 2 y 3 — Aglomerar y depurar: la segmentación completa


In [ ]:
t0 = time.time()
# Reusa el K-means ya ajustado (kmeansObj=km). minSegmentSize=50 px = 4.5 ha a 30 m.
res = shepseg.doShepherdSegmentation(
    img, minSegmentSize=50, imgNullVal=nodata, kmeansObj=km)
seg = res.segimg                       # imagen de etiquetas: a cada píxel, el id de su objeto

print(f"Segmentación completa en {time.time()-t0:.1f} s")
print(f"  · objetos finales            : {int(seg.max()):,}")
print(f"  · píxeles sueltos absorbidos : {res.singlePixelsEliminated:,}")
print(f"  · grupitos chicos fusionados : {res.smallSegmentsEliminated:,}")
print(f"  · umbral espectral de fusión : {res.maxSpectralDiff:.0f} (calculado automáticamente)")

## Visualicemos los objetos — acercándonos a la capital


In [ ]:
import matplotlib.patches as mpatches
cy, cx = 2266, 2029          # centro de la mancha urbana de la capital
V = 350                      # media ventana: 350 px = 10.5 km
r0, r1, c0, c1 = cy - V, cy + V, cx - V, cx + V

bordes = (ndimage.maximum_filter(seg[r0:r1, c0:c1], size=2)
          != ndimage.minimum_filter(seg[r0:r1, c0:c1], size=2))
vis = rgb[r0:r1, c0:c1].copy()
vis[bordes] = [1, 1, 0]      # fronteras en amarillo

fig, ax = plt.subplots(1, 2, figsize=(12.5, 6))
ax[0].imshow(rgb[::3, ::3])
ax[0].add_patch(mpatches.Rectangle((c0/3, r0/3), (c1-c0)/3, (r1-r0)/3,
                                   fill=False, edgecolor="yellow", linewidth=2))
ax[0].set_title("El estado (recuadro = acercamiento)"); ax[0].axis("off")
ax[1].imshow(vis); ax[1].set_title(f"{int(seg.max()):,} objetos — detalle: la capital"); ax[1].axis("off")
plt.tight_layout(); plt.show()
print("💡 Ya no son píxeles: son parcelas, manzanas, presas. Objetos con sentido.")

## De la imagen a la tabla — estadística zonal por objeto


In [ ]:
import pandas as pd
t0 = time.time()
nseg = int(seg.max()) + 1
flat = seg.ravel()
n_px = np.bincount(flat, minlength=nseg)        # nº de píxeles por objeto = su tamaño

tabla = {"segment_id": np.arange(1, nseg), "n_px": n_px[1:]}
den = np.where(n_px > 0, n_px, 1)               # denominador seguro (el objeto 0 = "sin dato")
# Estadística zonal: media y desviación de cada banda por objeto, en una pasada con bincount.
for b in range(n_bandas):
    v = img[b].ravel().astype(np.float64)
    suma  = np.bincount(flat, weights=v,     minlength=nseg)
    suma2 = np.bincount(flat, weights=v * v, minlength=nseg)
    media = suma / den
    var   = np.maximum(suma2 / den - media**2, 0)
    tabla[f"b{b+1}Mean"]   = media[1:]          # característica: media de la banda en el objeto
    tabla[f"b{b+1}StdDev"] = np.sqrt(var)[1:]   # característica: variabilidad interna del objeto

df = pd.DataFrame(tabla)
print(f"Tabla en {time.time()-t0:.2f} s: {df.shape[0]:,} objetos × {df.shape[1]-2} características")
df.head()

## Y de regreso al mapa — cada objeto por su infrarrojo cercano (NIR)


In [ ]:
# Pintar cada objeto por su NIR medio (vigor de vegetación) con una tabla de búsqueda
lut_nir = np.zeros(nseg, dtype=np.float32)
lut_nir[df.segment_id.values] = df.b4Mean.values     # banda 4 = NIR
mapa_nir = np.where(seg > 0, lut_nir[seg], np.nan)

plt.figure(figsize=(7.8, 7))
plt.imshow(mapa_nir[::3, ::3], cmap="RdYlGn")
plt.colorbar(shrink=0.7, label="NIR medio del objeto")
plt.title("Cada objeto pintado por su infrarrojo cercano (verde = vegetación vigorosa)")
plt.axis("off"); plt.show()

## La verdad-terreno la construimos AQUÍ (su linaje completo)

No viene pre-hecha: partimos de las **localidades urbanas del Marco Geoestadístico (INEGI)** y las convertimos en una máscara alineada a la imagen.


In [ ]:
import geopandas as gpd
# Localidades del Marco Geoestadístico (INEGI) en el encuadre (GeoJSON, WGS84)
ruta_loc = carga_dato("localidades_encuadre.geojson")
loc = gpd.read_file(ruta_loc)
print(f"{len(loc)} localidades en el encuadre")

# Linaje 1: filtrar las URBANAS
urbanas = loc[loc.AMBITO == "Urbana"]
print(f"→ {len(urbanas)} localidades urbanas")
# Linaje 2: proyectarlas al sistema de coordenadas de la imagen
urbanas = urbanas.to_crs(crs)
# Linaje 3: rasterizarlas a la malla exacta de 30 m
etiquetas = features.rasterize(
    ((g, 1) for g in urbanas.geometry),
    out_shape=seg.shape, transform=transform, fill=2, dtype="uint8")   # 1=urbano, 2=resto

plt.figure(figsize=(7.8, 7))
plt.imshow(rgb[::3, ::3])
plt.imshow(np.where(etiquetas[::3, ::3] == 1, 1.0, np.nan),
           cmap="autumn", alpha=0.9, interpolation="nearest")
plt.title(f"Verdad-terreno: {len(urbanas)} localidades urbanas (naranja)")
plt.axis("off"); plt.show()

# Linaje 4: proporción urbana de cada objeto (conecta etiquetas con objetos)
urb_px = np.bincount(flat, weights=(etiquetas == 1).ravel(), minlength=nseg)
df["prop_urbano"] = (urb_px / den)[1:]          # den (denominador seguro) viene de la celda anterior
print(f"objetos mayormente urbanos   : {(df.prop_urbano >= 0.5).sum():,}")
print(f"objetos mayormente no urbanos: {(df.prop_urbano < 0.5).sum():,}")

## Entrenamiento — el mismo pipeline del curso

Escalar las características → una red neuronal cuya opinión se apila como features extra → un bosque de árboles que vota la clase final.


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

class StackingEstimator(BaseEstimator, TransformerMixin):
    """Apila las predicciones de un modelo como features extra para el siguiente."""
    def __init__(self, estimator):
        self.estimator = estimator
    def fit(self, X, y=None, **kw):
        self.estimator_ = clone(self.estimator); self.estimator_.fit(X, y, **kw); return self
    def transform(self, X):
        X = np.asarray(X); out = [X]
        if hasattr(self.estimator_, "predict_proba"):
            out.append(self.estimator_.predict_proba(X))
        out.append(self.estimator_.predict(X).reshape(-1, 1))
        return np.hstack(out)

# 1) Solo objetos PUROS para entrenar (>=90% de una clase): etiquetas confiables
puros = df[(df.prop_urbano >= 0.9) | (df.prop_urbano <= 0.1)].copy()
puros["clase"] = np.where(puros.prop_urbano >= 0.9, 1, 2)
# 2) Balancear: mismo número de ejemplos por clase (muestreo reproducible)
n_min = int(puros.clase.value_counts().min())
balanceado = pd.concat([
    puros[puros.clase == 1].sample(n=n_min, random_state=42),
    puros[puros.clase == 2].sample(n=n_min, random_state=42)])

feat_cols = [c for c in df.columns if c.startswith("b")]   # 12 características: media y desv. de las 6 bandas
X, y = balanceado[feat_cols], balanceado["clase"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)  # 70% entrena / 30% examen

t0 = time.time()
pipeline = make_pipeline(
    StandardScaler(),
    StackingEstimator(MLPClassifier(alpha=0.01, learning_rate_init=0.001,
                                    max_iter=300, random_state=42)),
    ExtraTreesClassifier(bootstrap=False, criterion="entropy", max_features=0.8,
                         n_estimators=100, random_state=42))
pipeline.fit(Xtr, ytr)
acc = accuracy_score(yte, pipeline.predict(Xte))   # exactitud = % de aciertos en el examen (30% no visto)
print(f"Entrenado en {time.time()-t0:.1f} s con {len(Xtr)} objetos ({n_min} por clase)")
print(f"Exactitud en prueba: {acc:.1%}")
ConfusionMatrixDisplay(confusion_matrix(yte, pipeline.predict(Xte)),
                       display_labels=["urbano", "no urbano"]).plot(cmap="Blues")
plt.title("Matriz de confusión (conjunto de prueba)"); plt.tight_layout(); plt.show()

## El gran final — clasificar TODO el estado y pintar el mapa


In [ ]:
from matplotlib.colors import ListedColormap
t0 = time.time()
df["pred"] = pipeline.predict(df[feat_cols])   # aplica el modelo a TODOS los objetos (incluidos los ambiguos)
print(f"✓ clasificados {len(df):,} objetos en {time.time()-t0:.1f} s")

lut_pred = np.zeros(nseg, dtype=np.uint8)
lut_pred[df.segment_id.values] = df.pred.values
mapa_pred = lut_pred[seg]                        # 1=urbano, 2=no urbano (0=sin dato)

fig, ax = plt.subplots(figsize=(8.2, 7.5))
ax.imshow(rgb[::3, ::3])
ax.imshow(np.where(mapa_pred[::3, ::3] == 1, 1.0, np.nan),
          cmap=ListedColormap(["#d62728"]), alpha=0.9, interpolation="nearest")
ax.set_title("Lo urbano de Aguascalientes según TU modelo (rojo)")
ax.axis("off")
plt.savefig("mapa_clasificado.png", dpi=130, bbox_inches="tight"); plt.show()

n_u = int((df.pred == 1).sum())
km2 = float((mapa_pred == 1).sum()) * 900 / 1e6  # cada píxel = 30×30 m = 900 m²; /1e6 → km²
print(f"objetos urbanos: {n_u:,}  |  superficie urbana estimada: {km2:.0f} km²")

## 📦 Tus productos, listos para QGIS

Cuatro archivos georreferenciados; al final se descargan a tu equipo.


In [ ]:
import geopandas as gpd
# 1) ráster clasificado del estado (GeoTIFF)
with rasterio.open("mapa_urbano.tif", "w", driver="GTiff",
                   height=mapa_pred.shape[0], width=mapa_pred.shape[1],
                   count=1, dtype="uint8", crs=crs, transform=transform, nodata=0) as dst:
    dst.write(mapa_pred, 1)
# 2) tabla de características + predicción (CSV)
df.round(3).to_csv("features_urbano.csv", index=False)
# 3) lo URBANO como polígonos (GeoJSON para QGIS)
mask_urb = (mapa_pred == 1).astype(np.uint8)
geoms = [{"properties": {"clase": "urbano"}, "geometry": g}
         for g, v in features.shapes(mask_urb, mask=mask_urb.astype(bool), transform=transform)]
gdf_urb = gpd.GeoDataFrame.from_features(geoms, crs=crs)
gdf_urb.to_file("urbano_aguascalientes.geojson", driver="GeoJSON")

productos = ["mapa_urbano.tif", "features_urbano.csv",
             "urbano_aguascalientes.geojson", "mapa_clasificado.png"]
for p in productos:
    print(f"  ✓ {p}  ({os.path.getsize(p)/1024:,.0f} KB)")

# Descargarlos a tu equipo (en Colab abre el diálogo de descarga)
try:
    from google.colab import files
    for p in productos:
        files.download(p)
except Exception:
    print("Si no se descargan solos, abre el panel de archivos (📁 a la izquierda) y bájalos.")

## 🧪 Experimenta tú

Cambia los dos parámetros clave sobre el acercamiento a la capital (toma segundos) y observa cómo cambia el mapa de objetos.

- `MI_TAMANO_MINIMO` — tamaño mínimo de objeto en píxeles. ¿Qué pasa con 10? ¿Y con 200?
- `MIS_CLUSTERS` — cuántas "familias espectrales" busca el paso 1. ¿Con 15? ¿Con 100?


In [ ]:
MIS_CLUSTERS = 30
MI_TAMANO_MINIMO = 100

ventana = img[:, r0:r1, c0:c1]        # el acercamiento a la capital
t0 = time.time()
mi_res = shepseg.doShepherdSegmentation(
    ventana, numClusters=MIS_CLUSTERS, minSegmentSize=MI_TAMANO_MINIMO,
    imgNullVal=nodata, fixedKMeansInit=True)
mi_seg = mi_res.segimg

bordes = (ndimage.maximum_filter(mi_seg, size=2) != ndimage.minimum_filter(mi_seg, size=2))
vis = rgb[r0:r1, c0:c1].copy(); vis[bordes] = [0, 1, 1]
plt.figure(figsize=(7, 7)); plt.imshow(vis)
plt.title(f"numClusters={MIS_CLUSTERS}, minSegmentSize={MI_TAMANO_MINIMO} → "
          f"{int(mi_seg.max()):,} objetos ({time.time()-t0:.1f} s)")
plt.axis("off"); plt.show()

## 🎓 Lo que acabas de lograr

1. Procesaste **todo el estado de Aguascalientes** (12 millones de píxeles) con Python científico real en la nube.
2. Segmentaste la imagen en objetos con `pyshepseg` (el mismo algoritmo del curso) y los describiste con **estadística zonal**.
3. Construiste la **verdad-terreno** desde las localidades del Marco Geoestadístico (INEGI) y entrenaste una **red neuronal apilada con árboles de decisión**.
4. Produjiste el **mapa urbano estatal** y lo exportaste a formatos que QGIS abre directo.

**¿Y la versión en el navegador?** El mismo taller corre sin instalar nada en `shepherd-wasm` (WebAssembly). El **laboratorio portable (SatLab)** lo instala en tu máquina sin límites de memoria, con la geomediana Sentinel-2 completa (10 m, 12 bandas).

---
*Segmentación: `pyshepseg` (ubarsc) — `shepherd-wasm` es su port a WebAssembly. Datos: geomediana Sentinel-2 remuestreada a 30 m / 6 bandas; localidades del Marco Geoestadístico (INEGI). Código: [github.com/abxda/portable-satelital](https://github.com/abxda/portable-satelital).*
